XGBoost In-Class Example

In [1]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pip install XGBoost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
default_df = pd.read_csv(r'c:\Users\prchandr\Downloads\creditcard.csv')
default_df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


## Performing initial EDA 

In [4]:
report = sv.analyze(default_df)
report.show_html('sweet_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


From the EDA visuals, we can see that most customers have a payment amounts that are low to moderate values. Most customers tend to pay on time or with minimal delay. However there is a noticeable different in behavior among one group for it. Customer demographic variables such as education, marital status, and gender were mostly evenly split. Additionally, we can see that most customers do not default which may affect predicitive modeling later on. The EDA reveal that around 50% of participant information has been married twice and has a majority male class (assuming sex=2, refers to males in this dataset). The ages center from 25-30 in the dataset, however there is a range from young adults (20 year olds) to older folk (60 years).

In [5]:
default = default_df.copy()

## Random Forest

In [6]:
X = default.drop('default payment next month', axis=1)
y = default['default payment next month']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3720
           1       0.64      0.38      0.47      1080

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest Confusion Matrix:
[[3490  230]
 [ 675  405]]
Random Forest Accuracy: 0.8114583333333333


In [13]:
# buildling baseline models with cross validation
rf_cv_scores = cross_val_score(rf_model, X, y, cv=5)
xgb_cv_scores = cross_val_score(xgb_model, X, y, cv=5)
print("Random Forest CV Scores:", rf_cv_scores)
print("XGBoost CV Scores:", xgb_cv_scores)
print("Random Forest CV Mean Score:", np.mean(rf_cv_scores))
print("XGBoost CV Mean Score:", np.mean(xgb_cv_scores))

Random Forest CV Scores: [0.8125     0.81416667 0.815      0.80916667 0.81875   ]
XGBoost CV Scores: [0.80979167 0.81083333 0.814375   0.81020833 0.81895833]
Random Forest CV Mean Score: 0.8139166666666666
XGBoost CV Mean Score: 0.8128333333333334


The mean and standard deviation represent different subsets of our customer data to make sure the results are reliable and not just based on one specific sample.

The average score tells us how well each model performs over all subsets. In this case, both models performed very similarly, with the Random Forest model having a slightly higher average performance. The variation in the scores across the tests is also very small for both models, which is a good sign. Neither model is overly sensitive to specific customer segments or patterns, and there is no evidence that results are being driven by a “lucky” or unusually favorable subset of data.

## Classification of Random Forest after Cross Validation using Hyperparameters

Look at the classification report for your two models. Does this change your evaluation of the models? 

In [17]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt'],
    'min_samples_leaf': [1, 2]
}

gs = GridSearchCV(estimator=rf_model, param_grid=param_grid_rf, cv=5, n_jobs=-1, verbose=2)
gs.fit(X_train, y_train)

rf_best = gs.best_estimator_
rf_pred = rf_best.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3738
           1       0.66      0.36      0.46      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.65      0.68      4800
weighted avg       0.80      0.82      0.80      4800



XGBoost Model

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

XGBoost Confusion Matrix:
[[3483  255]
 [ 656  406]]


Hypermeters for XGBoost

In [21]:
# hyperparameters for xgboost using 4 parameters
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8]
}
# Create a fresh estimator for GridSearchCV (avoids cloning issues with a fitted estimator)
grid_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid=param_grid_xgb,
    cv=5,
    n_jobs=-1,
    verbose=2
)
grid_xgb.fit(X_train.astype("float64").to_numpy(), y_train.astype("int64").to_numpy())
print("Best parameters for XGBoost:", grid_xgb.best_params_)
best_xgb = grid_xgb.best_estimator_
y_pred_xgb_tuned = best_xgb.predict(X_test.astype("float64").to_numpy())
print("Tuned XGBoost Classification Report:")
print(classification_report(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))
print("Tuned XGBoost Confusion Matrix:")
print(confusion_matrix(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))
print("Tuned XGBoost Accuracy:", accuracy_score(y_test.astype("int64").to_numpy(), y_pred_xgb_tuned))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters for XGBoost: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Tuned XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3738
           1       0.67      0.37      0.48      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.66      0.68      4800
weighted avg       0.80      0.82      0.80      4800

Tuned XGBoost Confusion Matrix:
[[3540  198]
 [ 667  395]]
Tuned XGBoost Accuracy: 0.8197916666666667


Considering class imbalances

In [20]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compute imbalance ratio for XGBoost
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

# Random Forest with class imbalance adjustment
rf_model_balanced = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

# XGBoost with class imbalance adjustment
xgb_model_balanced = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

# Cross-validation
rf_cv_scores_balanced = cross_val_score(rf_model_balanced, X, y, cv=cv)
xgb_cv_scores_balanced = cross_val_score(xgb_model_balanced, X, y, cv=cv)

print("Balanced Random Forest CV Scores:", rf_cv_scores_balanced)
print("Balanced XGBoost CV Scores:", xgb_cv_scores_balanced)
print("Balanced Random Forest CV Mean Score:", np.mean(rf_cv_scores_balanced))
print("Balanced XGBoost CV Mean Score:", np.mean(xgb_cv_scores_balanced))

Balanced Random Forest CV Scores: [0.81604167 0.81104167 0.81604167 0.81520833 0.814375  ]
Balanced XGBoost CV Scores: [0.76       0.76604167 0.761875   0.764375   0.75395833]
Balanced Random Forest CV Mean Score: 0.8145416666666666
Balanced XGBoost CV Mean Score: 0.7612499999999999


## How did this impact model performance?

The Random Forest model maintained high and stable performance, with an average score of approximately 0.815, indicating that incorporating class weights did not significantly degrade its ability to generalize or capture underlying patterns in the data.

In contrast, the XGBoost model experienced a noticeable drop in performance, with its average score decreasing to approximately 0.761. This suggests that reweighting the minority class introduced a tradeoff, where the model became more sensitive to identifying default cases at the expense of overall accuracy. In other words, the XGBoost model placed greater emphasis on correctly identifying high-risk customers, even if it resulted in more errors on the majority class.

## Now try the XGBoost boostrapping (subsample =0.8) feature with. How did this affect performance?

When I applied subsampling (subsample = 0.8) to the XGBoost model, I forced the model to train on a smaller, random portion of the data in each iteration. This reduced the model’s tendency to rely too heavily on specific data points and helped control overfitting.

As a result, I observed more stable performance across validation sets, although the overall score may have changed slightly. I interpret this as the model becoming more robust and less sensitive to noise, even if it did not significantly increase overall accuracy.

## Do a brief tuning (2 or 3 features) for each model. Did your model performance increase or decrease? Do you think you chose the right parameters to tune?

I tuned a small set of key parameters for each model to see if I could improve performance without overcomplicating the model. For Random Forest, I increased `n_estimators` from 100 to 200 to improve stability, adjusted `max_depth` from 10 to 20 to allow the model to capture more complex patterns, and changed `min_samples_leaf` from 1 to 2 to reduce overfitting. For XGBoost, I reduced `learning_rate` from 0.3 to 0.1 to make learning more gradual, increased `max_depth` from 3 to 5 to capture additional structure, and applied `subsample = 0.8` to reduce over-reliance on specific data points.

After tuning, I observed that performance changed slightly but not dramatically. In some cases, performance improved marginally, while in others it decreased slightly, suggesting that the baseline models were already capturing most of the meaningful patterns in the data.

I think I chose the correct parameters because the dataset includes volatile balances and delays. Increasing depth allowed the models to better differentiate these high-risk patterns, while adding constraints like min_samples_leaf and subsampling helped prevent the model from overfitting to isolated or extreme cases.

## Out of the box, which model performed the best?

Out-of-the-box, the Random Forest model performed better than the XGBoost model. The Random Forest achieved a higher average cross-validation score (approximately 0.814) compared to XGBoost (approximately 0.813), and it also maintained stronger performance when accounting for class imbalance.